# PARC2026 ①マージ編 — LoRAアダプタを1つのモデルにする

**このノートの目的はたった2つです。**

1. 8/3に作った **LoRAアダプタ（8.7MB）** を、ベースモデルに**マージ**して1つのモデルにする
2. **`MyPolicy` を書くのに必要な情報**（モデルが何を入力に取るか）を調べる

**学習はしません。** 8/3に3000ステップ回した結果がDriveにあるので、それを使います。

---

## 前提

| | |
|---|---|
| **ランタイム** | **L4 GPU**（本番の採点環境と同じ。T4はbf16非対応なので不可） |
| Driveに置いてあるもの | `MyDrive/parc2026/smolvla_libero_plus_spatial_lora/checkpoints/003000/pretrained_model/adapter_model.safetensors` |
| 所要時間 | 15〜30分（ほぼダウンロード待ち） |

## 今どこにいるか

```
[済] Docker環境構築 → ランダムポリシーで提出 → スコア 0
                                                  ↓
[今ここ] マージ → モデルの入出力を調べる
                                                  ↓
[次]   MyPolicy を書く → ローカル評価で成功率>0 を確認 → 提出
                                                  ↓
       0 を脱出する  ← 8/14までの最優先目標
```

**成功率がゲートなので、1エピソードも成功しない限りスコアは必ず0です。** まずそこを越えます。

---
## 1. ランタイムの確認

**L4になっているか確認します。** 違っていたら `ランタイム → ランタイムのタイプを変更 → L4 GPU`。

⚠️ **途中で変えると `/content` が全部消えます。** ここで確定させてください。

In [ ]:
import subprocess, sys, torch
print(subprocess.run(["nvidia-smi","--query-gpu=name,memory.total","--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip())
print("torch:", torch.__version__, "/ CUDA:", torch.cuda.is_available())
print("bf16 使えるか:", torch.cuda.is_bf16_supported())
print()
print("→ 'NVIDIA L4' かつ bf16 True ならOK")

---
## 2. ライブラリのインストール

必要なのは2つだけです。

| | 役割 |
|---|---|
| **lerobot** | SmolVLAのモデル定義。`SmolVLAPolicy` がここにある |
| **peft** | LoRAを扱う。**マージの `merge_and_unload()` がここ** |

バージョンは公式ノートと同じ **v0.6.0** に固定します（違うと動かない可能性があります）。

⏱ 3〜5分かかります。

In [ ]:
%pip install -q "git+https://github.com/huggingface/lerobot.git@v0.6.0" peft safetensors
print("インストール完了")

In [ ]:
# --- 動作確認：ここが通らなければ先に進めない ---
try:
    import peft, lerobot
    from lerobot.configs import PreTrainedConfig
    from lerobot.policies.smolvla.modeling_smolvla import SmolVLAPolicy
    from peft import PeftModel
    print("✅ import 成功")
    print("  lerobot:", getattr(lerobot, "__version__", "不明"))
    print("  peft   :", peft.__version__)
except Exception as e:
    print("❌ import 失敗:", type(e).__name__, e)
    print()
    print("→ 公式ノート smolvla_libero_spatial_lora.ipynb の")
    print("  セクション1〜7 を先に実行してから、このノートの §3 以降を実行してください。")

In [ ]:
%pip install -q av

---
## 3. Driveをマウントして、アダプタを確認する

8/3に退避した**8.7MBのアダプタ**を探します。

**これが「あなたが34分かけて作った唯一のもの」**です。ベースモデルは公開されているので誰でも落とせますが、この `adapter_model.safetensors` だけは自分のものです。

In [ ]:
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')

ADAPTER_DIR = Path("/content/drive/MyDrive/parc2026/"
                   "smolvla_libero_plus_spatial_lora/checkpoints/003000/pretrained_model")

print("探す場所:", ADAPTER_DIR)
print()
if (ADAPTER_DIR / "adapter_model.safetensors").is_file():
    for f in sorted(ADAPTER_DIR.iterdir()):
        print(f"  {f.stat().st_size/1e6:8.2f} MB  {f.name}")
    print("\n✅ アダプタが見つかりました")
else:
    print("❌ 見つかりません。Drive の中を探します:")
    root = Path("/content/drive/MyDrive/parc2026")
    if root.exists():
        for f in root.rglob("adapter_model.safetensors"):
            print("  候補:", f.parent)
        print("\n→ 上のパスを ADAPTER_DIR に設定し直してください")
    else:
        print("  /content/drive/MyDrive/parc2026 が存在しません")

---
## 4. ベースモデルをダウンロード

`lerobot/smolvla_libero_plus` = **既にLIBERO-plusで学習済みのSmolVLA**（約450M）。

これが $W$ にあたります。**リビジョンを固定**しているのは、後から更新されて挙動が変わるのを防ぐためです（レポートにも「使用した重みのrevision」を書く欄があります）。

⏱ 2〜5分（約1GBのダウンロード）

In [ ]:
from huggingface_hub import snapshot_download

BASE_MODEL_REPO     = "lerobot/smolvla_libero_plus"
BASE_MODEL_REVISION = "7bb70aa5bc92b82c9239142775d3a173103567ff"
VLM_REPO            = "HuggingFaceTB/SmolVLM2-500M-Video-Instruct"

BASE_MODEL_LOCAL = Path(snapshot_download(
    repo_id=BASE_MODEL_REPO,
    revision=BASE_MODEL_REVISION,
    allow_patterns=[
        "config.json", "model.safetensors", "train_config.json",
        "policy_preprocessor.json", "policy_preprocessor*.safetensors",
        "policy_postprocessor.json", "policy_postprocessor*.safetensors",
    ],
))

print("保存先:", BASE_MODEL_LOCAL)
for f in sorted(BASE_MODEL_LOCAL.iterdir()):
    if f.is_file():
        print(f"  {f.stat().st_size/1e6:8.1f} MB  {f.name}")

---
## 5. マージ — $W + BA$ を計算する

**ここが本体です。** やっていることは3行です。

```python
base_policy   = SmolVLAPolicy.from_pretrained(...)   # W を読む
peft_policy   = PeftModel.from_pretrained(base, adapter)   # B と A を被せる
merged_policy = peft_policy.merge_and_unload()       # W + BA を計算して覆いを外す
```

$$W_{\text{新}} = W + BA$$

**行列の足し算なので数秒で終わります。** そしてサイズは増えません（8.7MBが「追加」されるのではなく、$W$ の数値に**足し込まれる**ため）。

マージ後に設定を書き換えているのは、**「これはもうLoRAではなく普通のモデルだ」と記録するため**です（`use_peft = False`）。

⚠️ CPUでマージします。GPUメモリを使わないので安全です。

In [ ]:
%pip install -q num2words accelerate datasets

In [ ]:
%pip uninstall -y torchao

In [ ]:
import gc, io, contextlib, shutil, torch
from safetensors import safe_open

MERGED_MODEL_DIR = Path("/content/merged_model")

merge_config = PreTrainedConfig.from_pretrained(ADAPTER_DIR)
merge_config.device          = "cpu"
merge_config.pretrained_path = BASE_MODEL_LOCAL
merge_config.use_peft        = False

gc.collect(); torch.cuda.empty_cache()

quiet = io.StringIO()
with contextlib.redirect_stdout(quiet), contextlib.redirect_stderr(quiet):
    base_policy   = SmolVLAPolicy.from_pretrained(
        BASE_MODEL_LOCAL, config=merge_config, strict=False)
    peft_policy   = PeftModel.from_pretrained(
        base_policy, ADAPTER_DIR, is_trainable=False, torch_device="cpu")
    merged_policy = peft_policy.merge_and_unload(safe_merge=True)

# 「もうLoRAではない」ことを設定に記録する
shutil.rmtree(MERGED_MODEL_DIR, ignore_errors=True)
MERGED_MODEL_DIR.mkdir(parents=True, exist_ok=True)
merged_policy.config.use_peft         = False
merged_policy.config.pretrained_path  = None
merged_policy.config.push_to_hub      = False
merged_policy.config.repo_id          = None
merged_policy.config.device           = None
merged_policy.config.load_vlm_weights = False
merged_policy.config.vlm_model_name   = VLM_REPO
merged_policy.save_pretrained(MERGED_MODEL_DIR)

# 前処理・後処理の設定ファイルも一緒に運ぶ（正規化のパラメータが入っている）
for pattern in ["policy_preprocessor.json", "policy_preprocessor*.safetensors",
                "policy_postprocessor.json", "policy_postprocessor*.safetensors"]:
    for src in ADAPTER_DIR.glob(pattern):
        shutil.copy2(src, MERGED_MODEL_DIR / src.name)

print("マージ完了:", MERGED_MODEL_DIR)
for f in sorted(MERGED_MODEL_DIR.iterdir()):
    if f.is_file():
        print(f"  {f.stat().st_size/1e6:8.1f} MB  {f.name}")

In [ ]:
# --- 検証：LoRAの痕跡が残っていないこと ---
w = MERGED_MODEL_DIR / "model.safetensors"
with safe_open(w, framework="pt", device="cpu") as f:
    keys = list(f.keys())
    lora_keys = [k for k in keys if "lora_" in k.lower()]

print(f"重みのキー数 : {len(keys)}")
print(f"lora_ を含む : {len(lora_keys)}")
print()
if lora_keys:
    print("❌ マージ失敗。LoRAのキーが残っています:", lora_keys[:5])
else:
    print("✅ マージ成功。BA は W に吸収され、普通のモデルになりました")

---
## 6. ★ モデルの入出力を調べる — 今日の本命

**`MyPolicy` を書くために必要な、唯一の未知がここです。**

PARCの `get_action(obs)` は次の辞書を受け取ります。

```
obs["agentview_image"]           (128,128,3) uint8   固定カメラ
obs["robot0_eye_in_hand_image"]  (128,128,3) uint8   手首カメラ
obs["robot0_joint_pos"]          (7,)   関節角
obs["robot0_eef_pos"]            (3,)   手先位置
obs["robot0_eef_quat"]           (4,)   手先姿勢
obs["robot0_gripper_qpos"]       (2,)   グリッパ開度
```

**これをSmolVLAが期待する形に変換する必要があります。** そのために「SmolVLAは何というキー名で、どんな形のデータを欲しがるのか」を調べます。

公式の評価設定から、すでに一部は分かっています。

| PARC側 | SmolVLA側 |
|---|---|
| `agentview_image` | **`front`** |
| `robot0_eye_in_hand_image` | **`wrist`** |
| 128×128 | ⚠️ **256×256を想定** → リサイズが要るかも |

In [ ]:
import inspect, json
policy = SmolVLAPolicy.from_pretrained(MERGED_MODEL_DIR)
policy.eval()

print("="*60)
print("【入力】モデルが期待するキーと形")
print("="*60)
for k, v in policy.config.input_features.items():
    print(f"  {k}")
    print(f"      type={getattr(v,'type',None)}  shape={getattr(v,'shape',None)}")

print()
print("="*60)
print("【出力】")
print("="*60)
for k, v in policy.config.output_features.items():
    print(f"  {k}")
    print(f"      type={getattr(v,'type',None)}  shape={getattr(v,'shape',None)}")

In [ ]:
print("="*60)
print("【推論の呼び方】select_action のシグネチャ")
print("="*60)
print(inspect.signature(policy.select_action))
print()
print("="*60)
print("【中身】action chunk を持っているか？")
print("="*60)
src = inspect.getsource(policy.select_action)
print(src[:2500])

In [ ]:
print("="*60)
print("【チャンク長など重要な設定】")
print("="*60)
for attr in ["n_action_steps", "chunk_size", "n_obs_steps",
             "max_state_dim", "max_action_dim",
             "resize_imgs_with_padding", "empty_cameras"]:
    if hasattr(policy.config, attr):
        print(f"  {attr:28s} = {getattr(policy.config, attr)}")

print()
print("→ n_action_steps > 1 なら、モデルが内部で action chunk を持っている")
print("   = 10秒制約への対策が半分すでに済んでいる")

In [ ]:
# observation.state の正規化統計を見る → 値の大きさで中身が分かる
import json, torch
from safetensors.torch import load_file

pre = MERGED_MODEL_DIR / "policy_preprocessor.json"
if pre.is_file():
    print(json.dumps(json.load(open(pre)), indent=2)[:3000])

for f in MERGED_MODEL_DIR.glob("policy_preprocessor*.safetensors"):
    print("\n===", f.name, "===")
    for k, v in load_file(f).items():
        if "state" in k.lower():
            print(f"{k}: {v.flatten().tolist()}")


---
## 7. マージ済みモデルをDriveへ保存

**ここまで来たら必ず保存してください。** ランタイムが切れると `/content` は消えます。

保存後、Windows側の `G:\マイドライブ\parc2026\merged_model` に同期されます。それを `PARC2026_pre/submission_template/model_weights/` にコピーすれば、提出物の準備が整います。

⏱ 約1GBのコピーなので数分かかります。**サイズが表示されるまで待ってください。**

In [ ]:
DRIVE_OUT = Path("/content/drive/MyDrive/parc2026/merged_model")
shutil.rmtree(DRIVE_OUT, ignore_errors=True)
DRIVE_OUT.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(MERGED_MODEL_DIR, DRIVE_OUT)

total = sum(f.stat().st_size for f in DRIVE_OUT.rglob("*") if f.is_file())
print(f"保存先: {DRIVE_OUT}")
print(f"合計   : {total/1e9:.2f} GB")
print()
for f in sorted(DRIVE_OUT.iterdir()):
    if f.is_file():
        print(f"  {f.stat().st_size/1e6:8.1f} MB  {f.name}")
print()
print("✅ ここまで出れば、ランタイムが切れても失うものはありません")

---
## 8. （任意）ダミー観測で推論を1回だけ試す

**PARCの `obs` を模したデータを作って、`select_action` が本当に動くか**を確かめます。

ここが通れば、`MyPolicy` を書く前に**入力の形が正しいことが確定**します。逆にここでエラーが出れば、そのエラーメッセージが「何をどう渡すべきか」を教えてくれます。

⚠️ **キー名は §6 の出力に合わせて書き換えてください。** 下は推測なので、そのままでは通らない可能性があります。**エラーを読むのが目的**です。

In [ ]:
import numpy as np, torch

# PARC の obs を模したダミー
dummy_obs = {
    "agentview_image":          np.random.randint(0,255,(128,128,3),dtype=np.uint8),
    "robot0_eye_in_hand_image": np.random.randint(0,255,(128,128,3),dtype=np.uint8),
    "robot0_joint_pos":    np.zeros(7, dtype=np.float32),
    "robot0_eef_pos":      np.zeros(3, dtype=np.float32),
    "robot0_eef_quat":     np.array([0,0,0,1], dtype=np.float32),
    "robot0_gripper_qpos": np.zeros(2, dtype=np.float32),
}

def to_chw(img):
    # (H,W,3) uint8 → (1,3,H,W) float32 [0,1]
    t = torch.from_numpy(img).float().permute(2,0,1) / 255.0
    return t.unsqueeze(0)

state = np.concatenate([dummy_obs["robot0_eef_pos"],
                        dummy_obs["robot0_eef_quat"],
                        dummy_obs["robot0_gripper_qpos"]]).astype(np.float32)

batch = {
    "observation.images.front": to_chw(dummy_obs["agentview_image"]),
    "observation.images.wrist": to_chw(dummy_obs["robot0_eye_in_hand_image"]),
    "observation.state":        torch.from_numpy(state).unsqueeze(0),
    "task":                     ["pick up the black bowl and place it on the plate"],
}

policy.reset()
try:
    with torch.no_grad():
        action = policy.select_action(batch)
    print("✅ 推論成功")
    print("  action.shape :", tuple(action.shape))
    print("  action       :", action.detach().cpu().numpy().ravel()[:8])
    print()
    print("→ この shape を (7,) に変換するのが MyPolicy の仕事")
except Exception as e:
    print("❌ エラー:", type(e).__name__)
    print(e)
    print()
    print("→ このエラーが「何をどう渡すべきか」を教えてくれます。")
    print("  §6 の input_features と見比べて、キー名・形・dtype を直してください。")

---
## 9. 次にやること

このノートで得られるもの:

| | |
|---|---|
| ✅ マージ済みモデル（Driveに保存済み） | `MyDrive/parc2026/merged_model` |
| ✅ **入力のキー名と形** | §6 の `input_features` |
| ✅ **出力の形** | §6 の `output_features` |
| ✅ **chunkを持つか** | §8 の `n_action_steps` |

### ローカルでの手順

```bash
# 1. Driveから model_weights/ へコピー
#    G:\マイドライブ\parc2026\merged_model
#      → C:\Users\odeod\dev\PARC2026_pre\submission_template\model_weights\

# 2. requirements.txt に追記
#    torch, transformers, lerobot, peft(不要なら省く), safetensors

# 3. MyPolicy を書く（§6の結果に合わせて）

# 4. ★ ローカル評価で成功率が 0 でないことを確認
docker run -it --rm -v /c/Users/odeod/dev/PARC2026_pre:/host parc2026
python submission_template/policy_server.py --port 8000 &
python -m pipeline --server-url http://localhost:8000 --track track1 --n-episodes 2

# 5. 成功率 > 0 を確認してから zip → validate → 提出
cd submission_template && zip -r ../submission.zip policy_server.py requirements.txt model_weights/ && cd ..
python validate_submission.py submission.zip
```

**4番が最重要です。** ローカル評価は無制限なので、**成功率が0でないことを確認してから提出**してください。0のまま出すと1日1回の枠を無駄にします。

### 目標

**8/14までに「0を脱出する」。** それだけです。滑らかさの最適化はその後。